# 05 · 离群检测 & 合成样本检测

**输入文件**：`../data/train.csv`、`../data/test.csv`  
**输出文件**：
- `../outputs/figures/05_*.png`
- `../outputs/anomaly_score_train.npy`（供 Codex T8 消融实验使用）
- `../outputs/anomaly_score_valid.npy`  

**随机种子**：42  
**注意**：StandardScaler 只在训练集 fit，验证集和测试集 transform，防止数据泄露

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

SEED = 42
np.random.seed(SEED)

FIG_DIR = Path('../outputs/figures')
OUT_DIR = Path('../outputs')
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})

## 1. 读取数据 & 预处理

In [2]:
train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

feature_cols = [c for c in train.columns if c.startswith('var_')]

X = train[feature_cols].values
y = train['target'].values
X_test_raw = test[feature_cols].values

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# scaler 只 fit 训练集
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_valid_s = scaler.transform(X_valid)
X_test_s  = scaler.transform(X_test_raw)

print(f'训练集: {X_train_s.shape}  验证集: {X_valid_s.shape}  测试集: {X_test_s.shape}')
print(f'训练集正例率: {y_train.mean():.3f}  验证集正例率: {y_valid.mean():.3f}')

训练集: (160000, 200)  验证集: (40000, 200)  测试集: (200000, 200)
训练集正例率: 0.100  验证集正例率: 0.101


## 2. Isolation Forest

In [3]:
iso = IsolationForest(
    n_estimators=300,
    contamination=0.1,
    random_state=SEED,
    n_jobs=-1
)
iso.fit(X_train_s)

# anomaly_score：值越大越异常（取负的 decision_function）
train_score = -iso.decision_function(X_train_s)
valid_score = -iso.decision_function(X_valid_s)

# 保存供 Codex T8 消融实验使用
np.save(OUT_DIR / 'anomaly_score_train.npy', train_score)
np.save(OUT_DIR / 'anomaly_score_valid.npy', valid_score)
print('anomaly_score_train.npy & anomaly_score_valid.npy 已保存')

anomaly_score_train.npy & anomaly_score_valid.npy 已保存


In [4]:
# 各分位区间的正例比例
df_iso = pd.DataFrame({'score': train_score, 'target': y_train})
df_iso['quantile'] = pd.qcut(df_iso['score'], q=10, labels=[f'Q{i}' for i in range(1, 11)])
pos_by_q = df_iso.groupby('quantile', observed=True)['target'].mean()
global_pos = y_train.mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 分位区间正例率
axes[0].plot(range(len(pos_by_q)), pos_by_q.values, marker='o', color='#DD8452', linewidth=2)
axes[0].axhline(global_pos, color='gray', linestyle='--', label=f'全局正例率 {global_pos:.3f}')
axes[0].set_xticks(range(len(pos_by_q)))
axes[0].set_xticklabels(pos_by_q.index, rotation=30)
axes[0].set_xlabel('Anomaly Score 分位区间（低→高）')
axes[0].set_ylabel('正例率 (target=1)')
axes[0].set_title('Isolation Forest：Anomaly Score 分位区间 vs 正例率')
axes[0].legend()

# anomaly_score 分布（正负样本对比）
for label, color, lbl in [(0, '#4C72B0', 'target=0'), (1, '#DD8452', 'target=1')]:
    sns.kdeplot(train_score[y_train == label], ax=axes[1], color=color, label=lbl, fill=True, alpha=0.35)
axes[1].set_xlabel('Anomaly Score')
axes[1].set_ylabel('密度')
axes[1].set_title('训练集 Anomaly Score 分布（正负样本对比）')
axes[1].legend()

sns.despine()
plt.tight_layout()
plt.savefig(FIG_DIR / '05_isolation_forest_analysis.png', bbox_inches='tight')
plt.show()
print('saved: 05_isolation_forest_analysis.png')

saved: 05_isolation_forest_analysis.png


## 3. LOF（全量训练集）

In [5]:
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.1, n_jobs=-1)
lof_labels = lof.fit_predict(X_train_s)     # -1=离群, 1=正常
lof_scores = -lof.negative_outlier_factor_  # 值越大越异常

outlier_mask = lof_labels == -1
normal_mask  = lof_labels ==  1
pos_outlier = y_train[outlier_mask].mean()
pos_normal  = y_train[normal_mask].mean()

print(f'LOF 全量样本数: {len(lof_labels):,}')
print(f'LOF 离群样本数: {outlier_mask.sum():,} | 正例率: {pos_outlier:.4f}')
print(f'LOF 正常样本数: {normal_mask.sum():,}  | 正例率: {pos_normal:.4f}')

LOF 全量样本数: 160,000
LOF 离群样本数: 16,000 | 正例率: 0.2947
LOF 正常样本数: 144,000  | 正例率: 0.0789


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# LOF 分数分布（正负样本对比）
for label, color, lbl in [(0, '#4C72B0', 'target=0'), (1, '#DD8452', 'target=1')]:
    sns.kdeplot(np.clip(lof_scores, None, np.percentile(lof_scores, 99))[y_train == label], ax=axes[0],
                color=color, label=lbl, fill=True, alpha=0.35)
axes[0].set_xlabel('LOF Score (99th percentile clipped)')
axes[0].set_ylabel('密度')
axes[0].set_title('LOF Score 分布（正负样本对比，全量训练集）')
axes[0].legend()

# 离群 vs 正常的正例率对比
axes[1].bar(['正常样本', '离群样本'], [pos_normal, pos_outlier],
            color=['#4C72B0', '#DD8452'], edgecolor='white', width=0.4)
axes[1].axhline(y_train.mean(), color='gray', linestyle='--',
                label=f'全量正例率 {y_train.mean():.3f}')
for i, v in enumerate([pos_normal, pos_outlier]):
    axes[1].text(i, v + 0.002, f'{v:.4f}', ha='center', fontsize=10)
axes[1].set_ylabel('正例率')
axes[1].set_title('LOF：离群 vs 正常样本的正例率')
axes[1].legend()

sns.despine()
plt.tight_layout()
plt.savefig(FIG_DIR / '05_lof_analysis.png', bbox_inches='tight')
plt.show()
print('saved: 05_lof_analysis.png')

saved: 05_lof_analysis.png


## 4. 合成样本检测（Santander 重点分析）

> PPT 明确指出："测试集藏有合成样本，先甄别真伪再建模是高分团队的关键。"  
> 本项目不以 Kaggle 排名为目标，因此只作为数据分布分析，不将测试集信息用于验证集指标评估。

In [7]:
test_score = -iso.decision_function(X_test_s)
test_labels = iso.predict(X_test_s)  # -1=疑似合成, 1=正常

n_synthetic = (test_labels == -1).sum()
print(f'测试集总样本: {len(test_labels):,}')
print(f'疑似合成样本: {n_synthetic:,}  ({n_synthetic/len(test_labels):.2%})')
print(f'疑似真实样本: {(test_labels==1).sum():,}  ({(test_labels==1).sum()/len(test_labels):.2%})')

测试集总样本: 200,000
疑似合成样本: 19,584  (9.79%)
疑似真实样本: 180,416  (90.21%)


In [8]:
# 训练集 vs 测试集 anomaly_score 分布对比图（关键图表）
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 核密度对比
sns.kdeplot(train_score, ax=axes[0], color='#4C72B0', label='Train', fill=True, alpha=0.4)
sns.kdeplot(test_score,  ax=axes[0], color='#DD8452', label='Test',  fill=True, alpha=0.4)
axes[0].set_xlabel('Anomaly Score')
axes[0].set_ylabel('密度')
axes[0].set_title('训练集 vs 测试集 Anomaly Score 分布对比')
axes[0].legend()

# 直方图对比（归一化）
axes[1].hist(train_score, bins=60, alpha=0.55, density=True, color='#4C72B0', label='Train')
axes[1].hist(test_score,  bins=60, alpha=0.55, density=True, color='#DD8452', label='Test')
axes[1].set_xlabel('Anomaly Score')
axes[1].set_ylabel('频率密度')
axes[1].set_title('训练集 vs 测试集 Anomaly Score 直方图')
axes[1].legend()

sns.despine()
plt.tight_layout()
plt.savefig(FIG_DIR / '05_train_test_anomaly_distribution.png', bbox_inches='tight')
plt.show()
print('saved: 05_train_test_anomaly_distribution.png')

saved: 05_train_test_anomaly_distribution.png


## 小结

**Isolation Forest 结果：**
- 高 anomaly_score 区间（Q8~Q10）的正例率明显高于全局均值，说明离群样本与交易正例存在一定正相关
- `anomaly_score` 已保存为 `.npy` 文件，供 Codex T8 消融实验使用

**LOF 结果（全量训练集 160k）：**
- LOF 检测到的离群样本正例率 vs 正常样本正例率，反映了局部密度视角下的异常与交易倾向关系

**合成样本检测：**
- 测试集中疑似合成样本约占 10%（contamination 参数设定）
- 训练集与测试集 anomaly_score 分布存在差异，测试集整体分布略有偏移，印证了测试集包含非训练分布样本的假设
- 本项目不调整建模策略，仅作为数据分布分析报告